# MRO 분석 — 중거리 규칙성 (Medium-Range Order)

비정질 구조의 **중거리 규칙성**을 트래젝토리 시간평균으로 분석합니다.
원소 비종속(`amorph`). 분석: 고리통계 · Bhatia-Thornton · 클러스터/퍼콜레이션 · 이면각 · 사면체 연결성.

**사용법**: *Config* 셀만 바꿔 위에서부터 실행. 고리/이면각은 비싸므로 `MAX_FRAMES_*`로 프레임 수를 제한합니다.

## 0. Import

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import amorph
from amorph import core, presets
from amorph.mro import rings, bhatia_thornton, clusters, dihedral, tetra_connectivity
from amorph.core.elements import element_colors
print('amorph', amorph.__version__)

## 1. Config — 여기 값만 바꾼다

- `FREE_ELEMENT` : 자유 클러스터로 볼 원소 (SiCN=자유탄소 'C').
- `NETWORK_GROUP` : 망 퍼콜레이션을 볼 원소군 (SiCN=Si+N).
- `BT_GROUP_A/B` : Bhatia-Thornton 의사이원 군 (망 vs 자유탄소).
- `TETRA_CENTER` : 사면체 중심 원소 (SiCN=Si).
- `MAX_FRAMES_RINGS/DIHEDRAL` : 비싼 분석의 프레임 상한 (균등 샘플링).

In [ ]:
TRAJ          = 'dump.300K.lammpstrj'
TYPE_MAP      = presets.SICN_TYPE_MAP
FIXED_CUTOFFS = presets.SICN_CUTOFFS

FREE_ELEMENT  = 'C'
NETWORK_GROUP = ['Si', 'N']
BT_GROUP_A    = ['Si', 'N']     # 망
BT_GROUP_B    = ['C']           # 자유탄소
TETRA_CENTER  = 'Si'

PROD_RANGE    = None
STRIDE        = 1
MAX_RING_SIZE = 12
MAX_FRAMES_RINGS    = 3      # 고리: 비싸므로 소수 프레임
MAX_FRAMES_DIHEDRAL = 3
NBLOCKS       = 5

## 2. Load + 시스템 정보

In [ ]:
frames_all = core.load(TRAJ, type_map=TYPE_MAP, frames='all')
traj = core.select_frames(frames_all, frame_range=PROD_RANGE, stride=STRIDE)
species = core.species_of(traj)
CM = core.CutoffMatrix(FIXED_CUTOFFS, default=0.0)
colors = element_colors(species)
print(f'production {len(traj)}/{len(frames_all)} 프레임, 원소 {species}, 원자수 {traj[0].n_atoms}')
rho = np.mean([fr.mass_density() for fr in traj])
print(f'질량밀도 = {rho:.4f} g/cc   cutoff: {CM}')

## 3. 고리 통계 (King's 최단경로)

결합망의 고리 크기 분포 + 조성(pure-X/mixed) 분해. 시간평균 ±1σ.

In [ ]:
RG = rings.ring_statistics(traj, CM, max_size=MAX_RING_SIZE,
                           max_frames=MAX_FRAMES_RINGS, n_blocks=min(NBLOCKS, MAX_FRAMES_RINGS))
sizes = RG['sizes']
fig, axes = plt.subplots(1,2, figsize=(14,4.8))
axes[0].bar(sizes, RG['count']['mean'], yerr=RG['count']['err'], color='#4a7ab8', edgecolor='k', capsize=3)
axes[0].set_xlabel('ring size'); axes[0].set_ylabel('rings / frame'); axes[0].set_title('Ring-size distribution (King)')
axes[0].set_xticks(sizes); axes[0].grid(alpha=0.3, axis='y')
# composition stack
classes = sorted({c for s in RG['composition'] for c in RG['composition'][s]})
bottom = np.zeros(len(sizes))
for cls in classes:
    vals = np.array([RG['composition'][int(s)].get(cls,0) for s in sizes])
    if vals.sum()==0: continue
    axes[1].bar(sizes, vals, bottom=bottom, label=cls, edgecolor='k', linewidth=0.3)
    bottom += vals
axes[1].set_xlabel('ring size'); axes[1].set_ylabel('rings (avg)'); axes[1].set_title('Ring composition'); axes[1].legend(fontsize=8); axes[1].set_xticks(sizes); axes[1].grid(alpha=0.3, axis='y')
plt.show()
print('rings/frame by size:', {int(s):round(float(m),1) for s,m in zip(sizes, RG['count']['mean']) if m>0.5})

## 4. Bhatia-Thornton — S_NN / S_CC / S_NC

의사이원 (망) vs (자유탄소). S_CC(Q→0) > c_A·c_B 이면 화학적 상분리(자유탄소 응집).

In [ ]:
BT = bhatia_thornton.bhatia_thornton(traj, BT_GROUP_A, BT_GROUP_B, r_max=10.0, nbins=500, n_blocks=NBLOCKS)
q = BT['q']
fig, axes = plt.subplots(1,3, figsize=(16,4.5))
axes[0].plot(q, BT['S_NN'], '#1f77b4'); axes[0].set_title('S_NN (topology)')
axes[1].plot(q, BT['S_CC'], '#d62728'); axes[1].axhline(BT['S_CC_ideal'], color='gray', ls='--', label=f"random {BT['S_CC_ideal']:.3f}")
axes[1].set_title('S_CC (chemistry)'); axes[1].legend(fontsize=9)
axes[2].plot(q, BT['S_NC'], '#2ca02c'); axes[2].axhline(0, color='gray', lw=0.6); axes[2].set_title('S_NC (cross)')
for ax in axes: ax.set_xlabel('Q (Å⁻¹)'); ax.grid(alpha=0.3); ax.axvspan(1.5,3.5, color='orange', alpha=0.06)
plt.suptitle(f"Bhatia-Thornton  A={BT['label_A']} (c={BT['c_A']:.3f})  B={BT['label_B']} (c={BT['c_B']:.3f})"); plt.show()
ilow = int(np.argmin(np.abs(q-0.5)))
msg = 'CLUSTERING(상분리)' if BT['S_CC'][ilow] > BT['S_CC_ideal'] else 'ordering(혼합 선호)'
print(f'S_CC(Q≈0.5)={BT["S_CC"][ilow]:.4f} vs ideal {BT["S_CC_ideal"]:.4f} → {msg}')

## 5. 클러스터 / 퍼콜레이션

자유탄소(C-C) 클러스터 크기·L_a·프랙탈차원 + 망(Si+N) 퍼콜레이션.

In [ ]:
FC = clusters.free_clusters(traj, CM, element=FREE_ELEMENT, n_blocks=NBLOCKS)
NET = clusters.network_percolation(traj, CM, group=NETWORK_GROUP, n_blocks=NBLOCKS)
print(f'자유-{FREE_ELEMENT}: 클러스터 {FC["n_clusters"][0]:.0f}±{FC["n_clusters"][1]:.0f}, '
      f'고립 {FC["n_isolated"][0]:.0f}, graphenic(≥{FC["min_graphenic"]}) {FC["n_graphenic"][0]:.0f}, '
      f'평균 L_a {FC["mean_La"][0]:.2f}±{FC["mean_La"][1]:.2f} Å, 최대크기 {FC["max_size"][0]:.0f}')
print(f'망({"+".join(NETWORK_GROUP)}) 최대성분 {NET["largest_fraction"][0]*100:.1f}%, '
      f'퍼콜레이션 확률 {NET["percolation_prob"][0]:.2f}, 축방향 점유 {NET["axial_fraction"][0]:.2f}')

recs = FC['last_frame']
Df, ndf = clusters.fractal_dimension(recs)
sizes = [r['size'] for r in recs]; Las = [r['L_a'] for r in recs if r['size']>=4]
fig, axes = plt.subplots(1,3, figsize=(16,4.5))
if sizes: axes[0].hist(sizes, bins=range(1,max(sizes)+2), color='#2ca02c', edgecolor='k')
axes[0].set_xlabel('cluster size'); axes[0].set_ylabel('count'); axes[0].set_title(f'free-{FREE_ELEMENT} sizes (last frame)'); axes[0].grid(alpha=0.3, axis='y')
if Las: axes[1].hist(Las, bins=20, color='#ff7f0e', edgecolor='k'); axes[1].axvline(np.mean(Las), color='r', ls='--', label=f'mean {np.mean(Las):.2f} Å'); axes[1].legend()
axes[1].set_xlabel('L_a (Å)'); axes[1].set_ylabel('count'); axes[1].set_title('L_a (size≥4)'); axes[1].grid(alpha=0.3, axis='y')
s=np.array([r['size'] for r in recs if r['size']>=3]); Rgs=np.array([r['R_g'] for r in recs if r['size']>=3])
if len(s): axes[2].scatter(Rgs, s, s=25, alpha=0.6, color='#1f77b4', edgecolor='k')
axes[2].set_xscale('log'); axes[2].set_yscale('log'); axes[2].set_xlabel('R_g (Å)'); axes[2].set_ylabel('size')
axes[2].set_title(f'mass-fractal  D_f={Df:.2f}  (n={ndf})'); axes[2].grid(alpha=0.3, which='both')
plt.show()

## 6. 이면각 (torsion) 분포

60°(gauche)·180°(trans) 부근 피크 = 형태 규칙성. 평탄 = IRO 없음.

In [ ]:
DH = dihedral.dihedral_distribution(traj, CM, nbins=36, max_frames=MAX_FRAMES_DIHEDRAL, n_blocks=min(NBLOCKS,MAX_FRAMES_DIHEDRAL))
fig, ax = plt.subplots(figsize=(9,5.5))
for lab, d in sorted(DH['dist'].items(), key=lambda kv:-DH['counts'].get(kv[0],0)):
    if np.all(np.isnan(d['p'])): continue
    ax.plot(DH['phi'], d['p'], lw=1.2, label=f'{lab} (n={DH["counts"].get(lab,0)})')
ax.axvline(60, color='green', ls=':', alpha=0.5); ax.axvline(180, color='orange', ls=':', alpha=0.5)
ax.set_xlabel('φ (deg)'); ax.set_ylabel('P(φ)'); ax.set_xlim(0,180); ax.set_title('Dihedral distribution (top types)')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3); plt.show()

## 7. 사면체 연결성 — corner / edge / face

중심 원소(예 Si) 사면체끼리 공유 원자 수로 분류. corner=1, edge=2, face=3.
α-Si₃N₄·β-SiC는 100% corner.

In [ ]:
TC = tetra_connectivity.tetra_connectivity(traj, CM, center=TETRA_CENTER, d_max=4.0, n_blocks=NBLOCKS)
cls = ['corner','edge','face','higher']
fig, axes = plt.subplots(1,2, figsize=(13,4.8))
means = [TC['by_conn'][c][0] for c in cls]; errs=[TC['by_conn'][c][1] for c in cls]
axes[0].bar(cls, means, yerr=errs, color=['#1f77b4','#ff7f0e','#d62728','#8c564b'], edgecolor='k', capsize=3)
for i,(m,c) in enumerate(zip(means,cls)):
    if m>0: axes[0].text(i, m, f'{TC["fractions"][c]*100:.1f}%', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel(f'{TETRA_CENTER}-{TETRA_CENTER} pairs / frame'); axes[0].set_title('Connectivity type'); axes[0].grid(alpha=0.3, axis='y')
ld = TC['last_frame']
for c,col in zip(['corner','edge','face'], ['#1f77b4','#ff7f0e','#d62728']):
    if ld.get(c): axes[1].hist(ld[c], bins=np.linspace(1.5,4.0,40), alpha=0.6, label=c, color=col)
axes[1].set_xlabel(f'{TETRA_CENTER}-{TETRA_CENTER} distance (Å)'); axes[1].set_ylabel('count'); axes[1].set_title('Distance by class'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.show()
br = {k:round(v[0],1) for k,v in TC['corner_bridge'].items() if v[0]>0}
print('corner 분율:', {c:round(TC['fractions'][c],3) for c in cls}, ' 브릿지 원소:', br)

---
MRO 분석 완료. 다른 온도/조성은 *Config*의 `TRAJ`만 바꿔 다시 실행하세요.